In [1]:
import numpy as np
import pandas as pd

import spice_net as spn

In [2]:
df = pd.read_csv('data/test_data.csv')
df

,init,result
0,-0.868245,-0.654525
1,0.866917,0.651528
2,-0.102037,-0.001062
3,0.602888,0.219135
4,0.838314,0.589141
...,...,...
2995,-0.181318,-0.005961
2996,0.314093,0.030987
2997,-0.313361,-0.030771
2998,0.144365,0.003009


In [3]:
som_size = 100

som_1 = spn.SpiceNetSom(n_neurons=som_size,
                        value_range_start=df['init'].min(),
                        value_range_end=df['init'].max(),
                        lrf_tuning_curve=spn.ConstLRF(0.8),
                        lrf_interaction_kernel=spn.ConstLRF(0.8))
som_2 = spn.SpiceNetSom(n_neurons=som_size,
                        value_range_start=df['result'].min(),
                        value_range_end=df['result'].max(),
                        lrf_tuning_curve=spn.ConstLRF(0.8),
                        lrf_interaction_kernel=spn.ConstLRF(0.8))

correlation_matrix = spn.SpiceNetHcm(som_1, som_2, spn.ConstLRF(0.8), spn.ConstLRF(0.8))

spice_net = spn.SpiceNet(correlation_matrix)

In [4]:
spice_net.fit(df['init'].tolist(), df['result'].tolist(), 10, 100, print_output=True)

100%|██████████| 30/30 [00:15<00:00,  1.96it/s]

Time spend on the Components: 
Som: 10.227823495864868 s | Convolution Matrix: 5.07023811340332 s


In [ ]:
# Print som neuron tuning curve width
min_width = 100
max_width = 0

for neuron in som_1.neurons:
    print(neuron.tuning_curve_width)
    if neuron.tuning_curve_width < min_width:
        min_width = neuron.tuning_curve_width
    if neuron.tuning_curve_width > max_width:
        max_width = neuron.tuning_curve_width
    
for neuron in som_2.neurons:
    print(neuron.tuning_curve_width)
    if neuron.tuning_curve_width < min_width:
        min_width = neuron.tuning_curve_width
    if neuron.tuning_curve_width > max_width:
        max_width = neuron.tuning_curve_width


0.005477914937687893
0.012115136181610982
0.01529054219192904
0.010477594675790935
0.011756690201644188
0.010774958226316982
0.01011066674862929
0.01214657750292603
0.015268418803512637
0.009940531832019551
0.009349618555133837
0.011228122763611728
0.012109906983826474
0.014529639297641644
0.01689022344938303
0.017418716572765718
0.02284032150561316
0.017159969241103704
0.012547569310494408
0.017356817634874618
0.01435259118157586
0.014097379082439434
0.011650381999429842
0.009459155643555388
0.014738392028123066
0.020331587483056843
0.021736454142641994
0.015650982354700537
0.014519488192976075
0.016011776225658164
0.009766147569995741
0.010623434953285764
0.012686824889385252
0.013418092110176368
0.012016021252161842
0.012708942166146361
0.01877378405573631
0.012832246678811847
0.00925704150531601
0.009112028271769523
0.010489995708579919
0.014415744953875114
0.012369924763835407
0.012044804114605083
0.009799397609051071
0.011379303089596136
0.011483131275046063
0.008278517837941026


In [6]:
min_width

np.float64(0.0017156157308674494)

In [7]:
from sinabs.spicenet.som_neuron import SpiceSOMNeuron

som_neuron = SpiceSOMNeuron(0.0017156157308674494, 1, 100)

In [8]:
import torch

results = som_neuron(torch.Tensor([0.1, 0.2, 1, 0.1]))
results

torch.Size([4, 3])
torch.Size([100, 4, 3])


tensor([[[ 4.5391e-02],
         [ 4.5391e-02],
         [ 4.5391e-02],
         [ 4.5391e-02]],

        [[ 4.5391e-02],
         [ 4.5391e-02],
         [ 4.5391e-02],
         [ 4.5391e-02]],

        [[ 4.5391e-02],
         [ 4.5391e-02],
         [ 4.5391e-02],
         [ 4.5391e-02]],

        [[ 4.5391e-02],
         [ 4.5391e-02],
         [ 1.6259e+00],
         [ 4.5391e-02]],

        [[ 4.5391e-02],
         [ 4.5391e-02],
         [ 1.8321e+00],
         [ 4.5391e-02]],

        [[-2.6741e-02],
         [-2.6741e-02],
         [ 7.3117e+00],
         [-2.6741e-02]],

        [[ 1.2165e-02],
         [ 4.5391e-02],
         [ 8.4084e+00],
         [ 1.2165e-02]],

        [[ 2.0081e-02],
         [-1.3145e-02],
         [ 1.8730e+00],
         [ 2.0081e-02]],

        [[-8.2032e-03],
         [-9.0491e-03],
         [ 3.4677e+00],
         [-8.2032e-03]],

        [[ 3.0085e-02],
         [ 5.0652e-02],
         [ 3.5817e+00],
         [ 3.0085e-02]],

        [[-2.1338e-0

In [9]:
def gauss_pdf(x, mu, sigma):
    return 1/(sigma * np.sqrt(2 * np.pi)) * np.exp( - (x - mu)**2 / (2 * sigma**2))

In [10]:
gauss_pdf(0.1, 1, 0.0017156157308674494)

np.float64(0.0)

In [11]:
gauss_pdf(0.2, 1, 0.0017156157308674494)

np.float64(0.0)

In [12]:
gauss_pdf(1, 1, 0.0017156157308674494)

np.float64(232.53591886786882)

In [13]:
# Average SNN Neuron results over last 50 time steps

result_one = results[50:, 0, :].mean().item()
result_two = results[50:, 1, :].mean().item()
result_three = results[50:, 2, :].mean().item()

In [14]:
result_one, result_two, result_three

(-0.060103364288806915, -0.08878672868013382, 4.1274518966674805)

In [15]:
0.1 / 0.0017156157308674494

58.28811091015003

In [16]:
result_one * 58.28811091015003, result_two * 58.28811091015003, result_three * 58

(-3.5033115637391283, -5.175210688657039, 239.39221000671387)

In [17]:
class SNNSom:
    def __init__(self, som, timesteps):
        self.som_size = len(som.neurons)
        self.timesteps = timesteps  
        
        self.neurons = []
        
        for neuron in som.neurons:
            self.neurons.append(SpiceSOMNeuron(neuron.tuning_curve_width, neuron.preferred_value, timesteps))
    
    def get_activation_vector(self, value):
        results = []
        
        if isinstance(value, list):
            for neuron in self.neurons:
                results_values = neuron(torch.Tensor(value))
                subresult = []
                for i in range(len(value)):
                    subresult.append(results_values[50:, i, :].mean().item())
                results.append(subresult)
        else:
            for neuron in self.neurons:
                results.append(neuron(torch.Tensor([value]))[50:, 0, :].mean().item())
        
        return results

In [18]:
snn_som = SNNSom(som=som_1, timesteps=100)

In [19]:
normal_som_result = som_1.get_activation_vector(0.4)
len(normal_som_result)

100

In [20]:
snn_som_response = snn_som.get_activation_vector(0.4)
len(snn_som_response)

torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size(

100

In [21]:
winner_neuron = np.argmax(snn_som_response)
winner_neuron

np.int64(72)

In [22]:
winner_neuron_normal_som = np.argmax(normal_som_result)
winner_neuron_normal_som

np.int64(71)

In [23]:
sorted_normal_response_with_index = sorted(enumerate(normal_som_result), key=lambda x: x[1], reverse=True)
sorted_snn_response_with_index = sorted(enumerate(snn_som_response), key=lambda x: x[1], reverse=True)

for i in range(100):
    print(sorted_normal_response_with_index[i], sorted_snn_response_with_index[i])

(71, np.float64(22.106624043160142)) (72, 4.204968452453613)
(72, np.float64(9.509298014529898)) (70, 4.090519428253174)
(70, np.float64(7.4767394432280065)) (71, 4.064678192138672)
(73, np.float64(0.9057188265060123)) (73, 4.0102410316467285)
(69, np.float64(0.3685873106819775)) (74, 3.8576877117156982)
(74, np.float64(0.017094873184707194)) (69, 3.853703022003174)
(68, np.float64(2.7546944547191967e-05)) (68, 3.3160147666931152)
(75, np.float64(1.5665379570792197e-07)) (75, 3.1419689655303955)
(66, np.float64(9.340751727746959e-12)) (67, 2.8905487060546875)
(67, np.float64(4.150156700801581e-12)) (76, 2.66568660736084)
(77, np.float64(1.2654178978534373e-13)) (77, 2.6563799381256104)
(76, np.float64(3.708426860284666e-16)) (66, 2.635861873626709)
(65, np.float64(2.5799769359993945e-16)) (65, 2.1880533695220947)
(64, np.float64(1.8324603890425476e-20)) (78, 2.1120188236236572)
(78, np.float64(3.1204512827853834e-22)) (64, 1.2448670864105225)
(79, np.float64(1.5052661960990416e-26)) (8

In [24]:
# Find matches in top 25
top_25_normal = [x[0] for x in sorted_normal_response_with_index[:25]]
top_25_snn = [x[0] for x in sorted_snn_response_with_index[:25]]

matches = 0

for i in range(25):
    if top_25_normal[i] in top_25_snn:
        matches += 1
matches

21

In [25]:
# Matches in top 50
top_50_normal = [x[0] for x in sorted_normal_response_with_index[:50]]
top_50_snn = [x[0] for x in sorted_snn_response_with_index[:50]]

matches = 0

for i in range(50):
    if top_50_normal[i] in top_50_snn:
        matches += 1
        
matches

29

In [26]:
# Exact matches top 25
matches = 0

for i in range(25):
    if sorted_normal_response_with_index[i][0] == sorted_snn_response_with_index[i][0]:
        matches += 1
matches

5

In [27]:
# For the first 100 init entry compare the winner neuron
winner_neurons_normal = []
winner_neurons_snn = []

for value in df['init'][:100]:
    winner_neurons_normal.append(np.argmax(som_1.get_activation_vector(value)))
    winner_neurons_snn.append(np.argmax(snn_som.get_activation_vector(value)))
    
# Do the same for som_2
snn_som_2 = SNNSom(som=som_2, timesteps=100)

winner_neurons_normal_2 = []
winner_neurons_snn_2 = []

for value in df['result'][:100]:
    winner_neurons_normal_2.append(np.argmax(som_2.get_activation_vector(value)))
    winner_neurons_snn_2.append(np.argmax(snn_som_2.get_activation_vector(value)))


torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size([1, 3])
torch.Size([100, 1, 3])
torch.Size(

In [28]:
# Get diff per entry between norm and snn
som_1_diff = [abs(x - y) for x, y in zip(winner_neurons_normal, winner_neurons_snn)]
som_2_diff = [abs(x - y) for x, y in zip(winner_neurons_normal_2, winner_neurons_snn_2)]

In [29]:
np.mean(som_1_diff), np.mean(som_2_diff)

(np.float64(0.64), np.float64(1.65))

In [36]:
# Now we do it batched

# For the first 100 init entry compare the winner neuron
winner_neurons_normal = []
winner_neurons_snn = []

for value in df['init'][:100]:
    winner_neurons_normal.append(np.argmax(som_1.get_activation_vector(value)))
print(df['init'][:100].to_list())
result = snn_som.get_activation_vector(df['init'][:100].to_list())
print(result)



[-0.8682447826250757, 0.866917431928073, -0.1020374904369925, 0.6028884400960157, 0.8383135643311708, 0.5096444988718252, -0.3071024726547451, -0.1370243087487783, -0.2508260279179017, -0.7856101621547595, -0.2236179997432437, 0.6559479066568092, 0.2505528473238594, 0.6744674309834453, -0.3776247958387329, 0.8496892275392962, -0.6371244540477354, -0.9409190105180608, 0.1213660705943369, 0.1866987750359596, 0.8139410266435063, -0.046475548908389, -0.1695601973054761, -0.954190116101316, -0.8532405462085937, 0.5979964827255073, -0.8442567872798592, -0.0580141505435471, -0.8557086534078719, 0.6806645380854803, -0.2535023133860417, 0.8812045723135709, 0.0721964052366668, -0.5524091777815572, -0.3719557870641672, -0.621141857530054, -0.1483048309712282, -0.2502121703214826, -0.5595615495074293, -0.489294717056197, -0.9637121602828806, -0.0016140491132725, -0.3087613030098239, -0.3623571873708642, -0.0808641360583914, 0.8060280503103872, -0.4587911747487518, 0.1510412466791866, 0.17426673069

In [ ]:
result_tensor = torch.Tensor(result).T
result_tensor.shape


torch.Size([100, 100])

In [37]:
winner_neurons_snn = torch.argmax(result_tensor, dim=1).tolist()
winner_neurons_snn

[7,
 94,
 46,
 83,
 92,
 76,
 35,
 43,
 36,
 12,
 37,
 85,
 65,
 86,
 31,
 94,
 19,
 2,
 58,
 60,
 90,
 51,
 41,
 0,
 8,
 83,
 8,
 50,
 7,
 87,
 36,
 95,
 54,
 21,
 31,
 19,
 43,
 38,
 22,
 25,
 1,
 52,
 34,
 31,
 47,
 91,
 26,
 59,
 60,
 9,
 78,
 32,
 60,
 24,
 38,
 57,
 8,
 14,
 82,
 94,
 56,
 66,
 72,
 91,
 94,
 83,
 72,
 56,
 5,
 63,
 71,
 36,
 98,
 53,
 32,
 2,
 74,
 30,
 59,
 56,
 66,
 16,
 9,
 60,
 43,
 53,
 20,
 91,
 65,
 52,
 36,
 84,
 63,
 65,
 16,
 38,
 66,
 4,
 93,
 36]

In [43]:
   
# Do the same for som_2
snn_som_2 = SNNSom(som=som_2, timesteps=100)

winner_neurons_normal_2 = []
winner_neurons_snn_2 = []

for value in df['result'][:100]:
    winner_neurons_normal_2.append(np.argmax(som_2.get_activation_vector(value)))
result_two = snn_som_2.get_activation_vector(df['result'][:100].to_list())

torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([1

In [44]:
result_two_tensor = torch.Tensor(result).T
print(result_two_tensor.shape)
winner_neurons_snn_2 = torch.argmax(result_two_tensor, dim=1).tolist()


torch.Size([100, 100])


In [45]:
# Get abs diff
som_1_diff = [abs(x - y) for x, y in zip(winner_neurons_normal, winner_neurons_snn)]
som_2_diff = [abs(x - y) for x, y in zip(winner_neurons_normal_2, winner_neurons_snn_2)]
print(som_1_diff)
print(som_2_diff)

[np.int64(2), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(2), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(2), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(2), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(2), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(2), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1)

In [46]:
# get mean diff
np.mean(som_1_diff), np.mean(som_2_diff)

(np.float64(0.58), np.float64(9.7))

In [7]:
import torch
from sinabs.spicenet.spice_som import SpiceSOM

std_list_som_1 = []
std_list_som_2 = []
pref_list_som_1 = []
pref_list_som_2 = []

for neuron in som_1.neurons:
    std_list_som_1.append(neuron.tuning_curve_width)
    pref_list_som_1.append(neuron.preferred_value)
    
for neuron in som_2.neurons:
    std_list_som_2.append(neuron.tuning_curve_width)
    pref_list_som_2.append(neuron.preferred_value)
    
som_1_spice_snn = SpiceSOM(std_list_som_1, pref_list_som_1, 100)
som_2_spice_snn = SpiceSOM(std_list_som_2, pref_list_som_2, 100)

In [9]:
result_som_1_test = som_1_spice_snn(torch.Tensor([0.4, 0.1]))

torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 100, 1])
torch.Size([100, 3])
torch.Size([100, 100, 3])
torch.Size([100, 100, 1])


In [10]:
result_som_1_test

tensor([[[[ 0.0505],
          [-0.3161],
          [-0.1874],
          ...,
          [ 0.0707],
          [ 0.1050],
          [-0.0603]],

         [[ 0.1168],
          [ 0.0291],
          [-0.0189],
          ...,
          [-0.0807],
          [ 0.1460],
          [-0.1427]],

         [[ 0.0061],
          [ 0.0752],
          [ 0.0104],
          ...,
          [-0.3327],
          [-0.0906],
          [ 0.0093]],

         ...,

         [[-0.0047],
          [-0.0399],
          [ 0.1005],
          ...,
          [-0.3612],
          [-0.0733],
          [-0.2151]],

         [[ 0.0662],
          [ 0.0454],
          [ 0.0329],
          ...,
          [ 0.0046],
          [-0.3242],
          [-0.0079]],

         [[ 0.1906],
          [-0.1298],
          [-0.1520],
          ...,
          [ 0.0454],
          [-0.0178],
          [-0.0477]]],


        [[[-0.0717],
          [ 0.1473],
          [ 0.1286],
          ...,
          [ 0.1077],
          [ 0.1164],
     

In [11]:
result_som_1_test.shape

torch.Size([2, 100, 100, 1])

In [47]:
result_som_1_test[1, 50:, 56, 0].mean().item()

4.3223443031311035

In [48]:
som_1.get_activation_vector(0.1)[56]

np.float64(12.154893413771845)